# Explore here

In [ ]:
# %% [markdown]
# # Muestreo Estratificado de 240 000 Registros Recientes con Dask

# %%
import dask.dataframe as dd
import pandas as pd

# Rutas completas
path_df1     = "/Users/joanafernandes/Data Science Bootcamp/Predicccion_agencia_de_viajes_proyecto/data/processed/dataset_completo.csv"
path_hoteles = "/Users/joanafernandes/Data Science Bootcamp/Predicccion_agencia_de_viajes_proyecto/data/processed/hotels_with_distance_dataset.csv"
path_vuelos  = "/Users/joanafernandes/Data Science Bootcamp/Predicccion_agencia_de_viajes_proyecto/data/processed/merged_flight_dataset.csv"

# Parámetros
fecha_col    = "fecha"
fecha_umbral = "2022-01-01"       # Dask acepta string ISO para fechas
N            = 240_000            # Tamaño de la muestra final

# Forzar dtypes de columnas inconsistentes en dataset_completo
dtype_overrides = {
    "evento_categoria": "object",
    "evento_desc":      "object",
    "evento_nombre":    "object"
}

# %%
# 1) Leer con Dask
ddf1 = dd.read_csv(
    path_df1,
    parse_dates=[fecha_col],
    assume_missing=True,
    dtype=dtype_overrides
)
ddf_hot  = dd.read_csv(path_hoteles, assume_missing=True)
ddf_vuel = dd.read_csv(path_vuelos,  assume_missing=True)

# 2) Filtrar por fecha y ordenar descendente
recent = ddf1[ddf1[fecha_col] >= fecha_umbral]
# Para ordenar por fecha, no es necesario set_index; haremos map_partitions
recent = recent.map_partitions(lambda df: df.sort_values(fecha_col, ascending=False))

# 3) Merge distribuido contra hoteles y vuelos
m1 = recent.merge(ddf_hot, how="left",
                  left_on="ciudad", right_on="destination_city")
m2 = m1.merge(ddf_vuel, how="left",
              left_on="ciudad", right_on="destination_city")

# 4) Calcular proporciones por ciudad (pequeño, compute en pandas)
counts = m2["ciudad"].value_counts().compute()
props  = counts / counts.sum()
n_city = (props * N).round().astype(int)

# 5) Definir función de muestreo estratificado en partición
def stratified_sample(df, n_dict):
    parts = []
    for city, n in n_dict.items():
        sub = df[df["ciudad"] == city]
        if len(sub) > 0:
            parts.append(sub.head(n))
    if parts:
        return pd.concat(parts)
    else:
        return df.iloc[0:0]

# 6) Aplicar muestreo en cada partición
sampled_dd = m2.map_partitions(stratified_sample, n_dict=n_city.to_dict())

# 7) Traer las primeras N filas ya muestreadas
sampled = sampled_dd.head(N, compute=True)

# 8) Guardar a disco
out1 = "/Users/joanafernandes/Data Science Bootcamp/Predicccion_agencia_de_viajes_proyecto/data/processed/total_data_240k.csv"
out2 = out1 + ".gz"
sampled.to_csv(out1, index=False)
sampled.to_csv(out2, index=False, compression="gzip")

print(f"✅ Guardado {len(sampled)} filas en:\n  • {out1}\n  • {out2}")